# Predicting the Electronic Band Gap of Inorganic Crystalline Materials from Composition and Structure
### Stage 8 — Explainability and Error Analysis

Uses the selected model from Stage 7: **ExtraTrees on composition+structure**. Since the large tree models weren't saved in Stage 6 (to avoid huge files), this notebook starts by refitting it on the exact same training rows and checking the result against Stage 6's saved test predictions, before computing:

1. Permutation importance (on the held-out test set)
2. SHAP values (on a documented subsample, for computational feasibility)
3. Partial dependence for the top descriptors
4. Error analysis: metals vs. non-metals, transition-metal-containing vs. main-group-only, error by band-gap range, error by crystal system, and the worst individual predictions

**What this notebook does not do:** it does not claim any feature "causes" the band gap — only that the model found it predictive. The chemical interpretation of these patterns is Stage 9.


## Load everything this stage needs

Four files: the composition+structure feature table (Stage 5), the raw cleaned dataset (Stage 4, for `chemsys`/`elements`/`crystal_system` — dropped from the one-hot feature table but needed here for grouping), the split ids (Stage 6), and the saved test predictions (Stage 6, for the integrity check).


In [ ]:
!pip install -q shap


In [ ]:
import os
import json
import ast
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

RANDOM_SEED = 42
N_TREES = 100  # must match Stage 6's non-FAST_MODE setting

DRIVE_DIR = "/content/drive/MyDrive/materials_bandgap_project"

def ensure_local(name):
    if os.path.exists(name):
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass
    drive_path = os.path.join(DRIVE_DIR, name)
    if os.path.exists(drive_path):
        shutil.copy(drive_path, name)
        print(f"Loaded {name} from Google Drive.")
    else:
        raise FileNotFoundError(f"{name} not found locally or at {drive_path}.")

for fname in ["mp_bandgap_features_composition_structure.csv", "mp_bandgap_stage4_cleaned.csv",
              "mp_bandgap_stage6_split_ids.csv", "stage6_test_predictions.csv"]:
    ensure_local(fname)

comp_struct = pd.read_csv("mp_bandgap_features_composition_structure.csv", keep_default_na=False, na_values=[""])
clean = pd.read_csv("mp_bandgap_stage4_cleaned.csv", keep_default_na=False, na_values=[""])
clean["elements"] = clean["elements"].apply(lambda v: ast.literal_eval(v) if isinstance(v, str) else v)
split_ids = pd.read_csv("mp_bandgap_stage6_split_ids.csv", keep_default_na=False, na_values=[""])
test_preds_saved = pd.read_csv("stage6_test_predictions.csv", keep_default_na=False, na_values=[""])

assert comp_struct["material_id"].is_unique and clean["material_id"].is_unique
print(f"Feature table: {comp_struct.shape} | Cleaned raw table: {clean.shape} | Split ids: {split_ids.shape}")


## Reconstruct the exact Stage 6 feature set, split, and complete-case mask


In [ ]:
ID_COLS = ["material_id", "formula_pretty", "band_gap", "energy_above_hull", "is_stable"]
FEATURE_COLS = [c for c in comp_struct.columns if c not in ID_COLS]
FORBIDDEN = {"band_gap", "cbm", "vbm", "efermi", "is_metal", "is_gap_direct",
             "energy_above_hull", "is_stable", "formation_energy_per_atom", "space_group_number"}
assert not (set(FEATURE_COLS) & FORBIDDEN), "Forbidden column in feature set"

data = comp_struct.merge(split_ids[["material_id", "grouped_split"]], on="material_id", validate="one_to_one")
y = data["band_gap"].to_numpy(dtype=float)
X = data[FEATURE_COLS].astype(float).to_numpy()
complete = ~np.isnan(X).any(axis=1)

train_mask = (data["grouped_split"] == "train").to_numpy()
test_mask = (data["grouped_split"] == "test").to_numpy()
train_idx = np.where(train_mask & complete)[0]
test_idx = np.where(test_mask & complete)[0]

print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Train rows (complete-case): {len(train_idx)} | Test rows (complete-case): {len(test_idx)}")


## Checkpointing (resume after a disconnect)

Every expensive step below (the refit, permutation importance, and SHAP) saves its result to a checkpoint folder on Drive as soon as it finishes. **If the runtime disconnects, reconnect and choose Runtime > Run all.** Finished steps are loaded from disk instead of being recomputed. Cheap steps (partial dependence, the error-analysis tables) are not checkpointed individually since they only take seconds once the model and predictions are available.


In [ ]:
import joblib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

CKPT_DIR = os.path.join(DRIVE_DIR if ON_COLAB else ".", "stage8_checkpoints", f"extratrees_seed{RANDOM_SEED}_trees{N_TREES}")
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoint directory:", CKPT_DIR)
print("Checkpoint files already present:", os.listdir(CKPT_DIR))


## Refit ExtraTrees and verify it matches Stage 6

Same hyperparameters, same training rows, same seed as Stage 6. If this doesn't reproduce Stage 6's saved predictions closely, something about the environment or data has changed since then, and that needs investigating before trusting anything downstream.


In [ ]:
MODEL_CKPT = os.path.join(CKPT_DIR, "extratrees_model.joblib")
PRED_CKPT = os.path.join(CKPT_DIR, "refit_test_predictions.npy")

if os.path.exists(MODEL_CKPT) and os.path.exists(PRED_CKPT):
    model = joblib.load(MODEL_CKPT)
    y_pred_test = np.load(PRED_CKPT)
    print(f"[resumed from checkpoint] loaded refit model and {len(y_pred_test)} test predictions")
else:
    model = ExtraTreesRegressor(
        n_estimators=N_TREES, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_SEED
    )
    t0 = time.time()
    model.fit(X[train_idx], y[train_idx])
    print(f"Refit in {time.time()-t0:.1f}s on {len(train_idx)} rows.")

    y_pred_test = model.predict(X[test_idx])
    joblib.dump(model, MODEL_CKPT)
    np.save(PRED_CKPT, y_pred_test)

saved = test_preds_saved[
    (test_preds_saved.feature_set == "composition_structure") & (test_preds_saved.model == "ExtraTrees")
].set_index("material_id")
check_ids = data["material_id"].to_numpy()[test_idx]
saved_aligned = saved.reindex(check_ids)

if saved_aligned["y_pred"].isna().any():
    print("WARNING: some refit test rows are not present in the Stage 6 saved predictions -- "
          "cannot fully verify the refit.")
else:
    diff = np.abs(y_pred_test - saved_aligned["y_pred"].to_numpy())
    print(f"Max |difference| vs Stage 6 saved predictions: {diff.max():.6f} eV")
    print(f"Mean |difference|: {diff.mean():.6f} eV")
    if diff.max() > 1e-6:
        print("NOTE: not bit-identical (can happen across sklearn/BLAS versions or thread "
              "scheduling in randomized ensembles) -- but should still be very close. "
              "Treat downstream results as approximate if this gap is not small.")
    else:
        print("Refit reproduces Stage 6 exactly.")

mae_check = np.abs(y_pred_test - y[test_idx]).mean()
print(f"\nRefit test MAE: {mae_check:.4f} eV (Stage 7 reported 0.5487 eV)")


## Disk space check

The permutation importance step below previously failed with `OSError: No space left on device`. That is Colab's local disk filling up -- `joblib`'s parallel backend memory-maps data to temporary files per worker, which can spike disk usage well above the data's actual size, especially with many CPU cores. Checking free space first, and listing the largest files in the working directory, so you can delete anything unneeded (old CSVs from earlier stages you no longer need in *this* session -- they are all safely on Drive already) before proceeding.


In [ ]:
!df -h /content 2>/dev/null || df -h .
print()
import subprocess
out = subprocess.run("du -ah . 2>/dev/null | sort -rh | head -20", shell=True, capture_output=True, text=True)
print("Largest files/folders in the current working directory:")
print(out.stdout)


## 1. Permutation Importance

Computed on the **held-out test set**, not the training set — permutation importance on training data can be inflated for models (like these tree ensembles) that overfit, since the model may be using a feature to memorize training-specific patterns rather than to generalize. Each feature's values are shuffled 10 times; the reported importance is the average increase in MAE caused by destroying that feature's information. This measures **predictive contribution**, not causation.


In [ ]:
PERM_CKPT = os.path.join(CKPT_DIR, "permutation_importance.csv")

if os.path.exists(PERM_CKPT):
    perm_df = pd.read_csv(PERM_CKPT)
    print(f"[resumed from checkpoint] loaded permutation importance for {len(perm_df)} features")
else:
    t0 = time.time()
    perm = permutation_importance(
        model, X[test_idx], y[test_idx],
        scoring="neg_mean_absolute_error", n_repeats=10, random_state=RANDOM_SEED,
        n_jobs=1,  # avoids joblib/loky memory-mapping the data to disk per worker;
                   # the model's own .predict() still uses multiple cores internally
    )
    print(f"Permutation importance computed in {time.time()-t0:.1f}s.")

    perm_df = pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance_mean_MAE_increase": perm.importances_mean,
        "importance_std": perm.importances_std,
    }).sort_values("importance_mean_MAE_increase", ascending=False).reset_index(drop=True)
    perm_df.to_csv(PERM_CKPT, index=False)

perm_df.to_csv("stage8_permutation_importance.csv", index=False)
print(perm_df.head(15).round(4).to_string(index=False))


In [ ]:
os.makedirs("figures", exist_ok=True)
top = perm_df.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(7.5, 6))
ax.barh(top["feature"], top["importance_mean_MAE_increase"],
        xerr=top["importance_std"], color="#4C72B0")
ax.set_xlabel("Increase in test MAE when shuffled (eV)")
ax.set_title("Permutation importance (top 15) -- ExtraTrees, composition+structure")
plt.tight_layout()
plt.savefig("figures/fig_stage8_permutation_importance.png", dpi=200, bbox_inches="tight")
plt.show()


## 2. SHAP Values

`TreeExplainer` computes exact SHAP values for tree ensembles efficiently, but summing per-tree contributions for every one of ~30,000 test rows across 100 trees is still substantial work. We use a **random subsample of 2,000 test rows** (fixed seed) rather than the full test set — this is stated explicitly so the SHAP results aren't mistaken for full-test-set coverage. Wrapped so a failure here (version mismatch, memory) doesn't block the rest of the notebook.


In [ ]:
SHAP_SAMPLE_SIZE = 2000
shap_ok = False

SHAP_VALUES_CKPT = os.path.join(CKPT_DIR, "shap_values.npy")
SHAP_SAMPLE_IDX_CKPT = os.path.join(CKPT_DIR, "shap_sample_idx.npy")
SHAP_IMPORTANCE_CKPT = os.path.join(CKPT_DIR, "shap_importance.csv")

try:
    import shap
    print("shap version:", shap.__version__)

    if os.path.exists(SHAP_VALUES_CKPT) and os.path.exists(SHAP_SAMPLE_IDX_CKPT) and os.path.exists(SHAP_IMPORTANCE_CKPT):
        shap_values = np.load(SHAP_VALUES_CKPT)
        shap_sample_idx = np.load(SHAP_SAMPLE_IDX_CKPT)
        mean_abs_shap = pd.read_csv(SHAP_IMPORTANCE_CKPT)
        print(f"[resumed from checkpoint] loaded SHAP values for {len(shap_sample_idx)} test rows")
    else:
        rng = np.random.default_rng(RANDOM_SEED)
        shap_sample_idx = rng.choice(test_idx, size=min(SHAP_SAMPLE_SIZE, len(test_idx)), replace=False)

        explainer = shap.TreeExplainer(model)
        t0 = time.time()
        shap_values = explainer.shap_values(X[shap_sample_idx])
        print(f"SHAP values computed for {len(shap_sample_idx)} test rows in {time.time()-t0:.1f}s.")

        mean_abs_shap = pd.DataFrame({
            "feature": FEATURE_COLS,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
        }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

        np.save(SHAP_VALUES_CKPT, shap_values)
        np.save(SHAP_SAMPLE_IDX_CKPT, shap_sample_idx)
        mean_abs_shap.to_csv(SHAP_IMPORTANCE_CKPT, index=False)

    mean_abs_shap.to_csv("stage8_shap_importance.csv", index=False)
    print(mean_abs_shap.head(15).round(4).to_string(index=False))

    fig = plt.figure(figsize=(8, 6))
    shap.summary_plot(shap_values, X[shap_sample_idx], feature_names=FEATURE_COLS,
                       max_display=15, show=False)
    plt.title(f"SHAP summary (n={len(shap_sample_idx)} test-set sample)")
    plt.tight_layout()
    plt.savefig("figures/fig_stage8_shap_summary.png", dpi=200, bbox_inches="tight")
    plt.show()

    shap_ok = True
except Exception as e:
    print(f"SHAP step failed or was skipped: {type(e).__name__}: {e}")
    print("Continuing with permutation importance only.")


In [ ]:
if shap_ok:
    rank_perm = perm_df.set_index("feature")["importance_mean_MAE_increase"].rank(ascending=False)
    rank_shap = mean_abs_shap.set_index("feature")["mean_abs_shap"].rank(ascending=False)
    rank_compare = pd.DataFrame({"perm_rank": rank_perm, "shap_rank": rank_shap}).dropna()
    spearman = rank_compare["perm_rank"].corr(rank_compare["shap_rank"], method="spearman")
    print(f"Spearman correlation between permutation-importance ranking and SHAP ranking: {spearman:.3f}")
    print("(High agreement between two independent explainability methods is a useful "
          "consistency check; it doesn't make either method causal.)")
else:
    print("SHAP unavailable this run -- relying on permutation importance only.")


## 3. Partial Dependence

For the top 3 features by permutation importance: how does the model's average prediction change as that one feature varies, holding the modeling procedure's treatment of other features fixed? This shows association the model has learned, not a controlled chemical experiment.


In [ ]:
top3 = perm_df["feature"].head(3).tolist()
print("Partial dependence for:", top3)

fig, ax = plt.subplots(figsize=(13, 4))
PartialDependenceDisplay.from_estimator(
    model, X[train_idx], features=[FEATURE_COLS.index(f) for f in top3],
    feature_names=FEATURE_COLS, ax=ax,
)
plt.tight_layout()
plt.savefig("figures/fig_stage8_partial_dependence.png", dpi=200, bbox_inches="tight")
plt.show()


## 4. Error Analysis

Using the refit model's own test predictions (not Stage 6's saved ones), joined back to `chemsys`, `elements`, and `crystal_system` from the Stage 4 cleaned dataset for grouping. **Residual = predicted − true**, so a positive mean residual means overprediction and negative means underprediction. All comparisons below are descriptive; the "why" is Stage 9.


In [ ]:
D_BLOCK = set([
    "Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn",
    "Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd",
    "Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg",
])

err = pd.DataFrame({
    "material_id": data["material_id"].to_numpy()[test_idx],
    "y_true": y[test_idx],
    "y_pred": y_pred_test,
})
err["residual"] = err["y_pred"] - err["y_true"]
err["abs_error"] = err["residual"].abs()

err = err.merge(clean[["material_id", "chemsys", "elements", "crystal_system", "nelements"]],
                 on="material_id", how="left", validate="one_to_one")
err["is_tm_containing"] = err["elements"].apply(
    lambda els: bool(set(els) & D_BLOCK) if isinstance(els, list) else False
)

print(f"Overall test set (this refit): MAE={err['abs_error'].mean():.4f}, "
      f"mean residual (bias)={err['residual'].mean():.4f}")


In [ ]:
is_metal = err["y_true"] == 0
summary_metal = err.groupby(is_metal.map({True: "metallic (true gap = 0)", False: "non-metallic (true gap > 0)"})).agg(
    n=("abs_error", "size"), MAE=("abs_error", "mean"), mean_residual=("residual", "mean"),
)
print("Error by metal/non-metal:")
print(summary_metal.round(4))
summary_metal.to_csv("stage8_error_by_metallicity.csv")


In [ ]:
summary_tm = err.groupby("is_tm_containing").agg(
    n=("abs_error", "size"), MAE=("abs_error", "mean"), mean_residual=("residual", "mean"),
)
summary_tm.index = summary_tm.index.map({True: "transition-metal-containing", False: "main-group-only"})
print("Error by transition-metal content:")
print(summary_tm.round(4))
summary_tm.to_csv("stage8_error_by_tm_content.csv")


In [ ]:
bins = [-0.001, 0, 1, 2, 3, 4, 6, 100]
labels = ["0 (metal)", "(0, 1]", "(1, 2]", "(2, 3]", "(3, 4]", "(4, 6]", "> 6"]
err["gap_bin"] = pd.cut(err["y_true"], bins=bins, labels=labels)

summary_bin = err.groupby("gap_bin", observed=True).agg(
    n=("abs_error", "size"), MAE=("abs_error", "mean"), mean_residual=("residual", "mean"),
)
print("Error by true band-gap range (eV):")
print(summary_bin.round(4))
summary_bin.to_csv("stage8_error_by_bandgap_bin.csv")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(summary_bin.index.astype(str), summary_bin["mean_residual"], color="#C44E52")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Mean residual (predicted - true), eV")
ax.set_xlabel("True band gap range (eV)")
ax.set_title("Prediction bias by band-gap range")
plt.tight_layout()
plt.savefig("figures/fig_stage8_bias_by_bandgap_bin.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
import seaborn as sns

top_systems = err["crystal_system"].value_counts().head(7).index
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=err[err["crystal_system"].isin(top_systems)],
            x="crystal_system", y="residual", order=top_systems, showfliers=False, ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Residual (predicted - true), eV")
ax.set_xlabel("Crystal system (7 most common)")
ax.set_title("Prediction bias by crystal system")
plt.tight_layout()
plt.savefig("figures/fig_stage8_bias_by_crystal_system.png", dpi=200, bbox_inches="tight")
plt.show()

print(err.groupby("crystal_system", observed=True)["residual"].agg(["size", "mean", "std"]).round(4)
      .sort_values("size", ascending=False))


In [ ]:
worst = err.merge(clean[["material_id", "formula_pretty"]], on="material_id", how="left") \
    .sort_values("abs_error", ascending=False).head(20)
worst_cols = ["material_id", "formula_pretty", "chemsys", "crystal_system",
              "y_true", "y_pred", "residual", "abs_error", "is_tm_containing"]
print(worst[worst_cols].round(3).to_string(index=False))
worst[worst_cols].to_csv("stage8_worst_predictions.csv", index=False)


## Save outputs


In [ ]:
OUT_FILES = [
    "stage8_permutation_importance.csv", "stage8_error_by_metallicity.csv",
    "stage8_error_by_tm_content.csv", "stage8_error_by_bandgap_bin.csv",
    "stage8_worst_predictions.csv",
]
if shap_ok:
    OUT_FILES.append("stage8_shap_importance.csv")

FIG_FILES = ["figures/fig_stage8_permutation_importance.png", "figures/fig_stage8_partial_dependence.png",
             "figures/fig_stage8_bias_by_bandgap_bin.png", "figures/fig_stage8_bias_by_crystal_system.png"]
if shap_ok:
    FIG_FILES.append("figures/fig_stage8_shap_summary.png")

config = {
    "random_seed": RANDOM_SEED, "n_trees": N_TREES,
    "sklearn_version": sklearn.__version__,
    "refit_test_MAE": float(mae_check),
    "shap_ran": shap_ok, "shap_sample_size": SHAP_SAMPLE_SIZE if shap_ok else None,
    "n_test_rows_complete_case": int(len(test_idx)),
}
with open("stage8_config.json", "w") as f:
    json.dump(config, f, indent=2)
OUT_FILES.append("stage8_config.json")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(os.path.join(DRIVE_DIR, "figures"), exist_ok=True)
    for fname in OUT_FILES:
        shutil.copy(fname, os.path.join(DRIVE_DIR, fname))
        print(f"Copied {fname} to Drive")
    for fname in FIG_FILES:
        if os.path.exists(fname):
            shutil.copy(fname, os.path.join(DRIVE_DIR, "figures", os.path.basename(fname)))
            print(f"Copied {fname} to Drive")
except ImportError:
    print("Not running in Colab -- files remain in the local working directory.")


### Stage 8 ends here.

Feature importance (two independent methods where SHAP succeeded), partial dependence, and a multi-angle error breakdown are complete. **No chemical explanation of these patterns has been written yet** — that's Stage 9, which will draw on:
- the top permutation/SHAP features
- the metal vs. non-metal and transition-metal vs. main-group error gaps
- the band-gap-range bias table (does underprediction at high gap get worse, and by how much)
- the worst-20 predictions, read chemically

**Do not run Stage 9 code in this notebook.** Await confirmation before proceeding.
